In [1]:
#pip install lap filterpy norfair torchreid opencv-python

In [2]:
from ultralytics import YOLO
import cv2
from IPython.display import display, clear_output
from PIL import Image

# ---- Paths ----
weights = "/home/ubuntu/yolo_training/runs/detect/train/weights/best.pt"  # your trained weights
source  = "/home/ubuntu/scripts_/Video1_crop.mp4"                         # video file
output  = "output_botsort.mp4"

# ---- Load YOLO Model ----
model = YOLO(weights)

# ---- Run YOLO + BoT-SORT Tracker ----
# You can adjust conf/iou/imgsz/device as you wish
results = model.track(
    source=source,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    device=0,
    tracker="botsort.yaml",  # use your config
    save=True,               # save output video automatically
    show=False,              # set True to open a window
    verbose=False
)

# ---- Counting Example ----
# (If you want to count confirmed tracks during processing)
# The Ultralytics tracker uses track.is_confirmed() internally based on n_init.
# If you want to verify counts programmatically:
counts = {}
for frame_id, frame_res in enumerate(results):
    if not hasattr(frame_res, "boxes") or frame_res.boxes is None:
        continue
    for box in frame_res.boxes:
        track_id = int(box.id.item()) if box.id is not None else None
        if track_id is not None:
            counts[track_id] = counts.get(track_id, 0) + 1

# Count only those IDs seen in ≥3 consecutive frames (same as n_init=3)
valid_ids = [tid for tid, fcount in counts.items() if fcount >= 3]

print(f"\n✅ Total unique confirmed objects (≥3 frames): {len(valid_ids)}")
print(f"✅ Output video saved to: {output}")
